# Week 41, Day 1: Moving: reaching, smoothness and minimum jerk

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w041/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Position, velocity and acceleration

Write min_jerk(x0, xf, T, t) returning the position at times t. Velocity and acceleration are computed numerically for you.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def min_jerk(x0, xf, T, t):
    tau = t / T
    return ...

T = 0.6
t = np.linspace(0, T, 601)
x = min_jerk(0.0, 0.3, T, t)
v = np.gradient(x, t); a = np.gradient(v, t)
print(f"peak velocity {v.max():.3f} m/s at t = {t[np.argmax(v)]:.3f} s; average speed {0.3 / T:.3f} m/s")
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
for axis, y, name in zip(ax, (x, v, a), ("position (m)", "velocity (m/s)", "acceleration (m/s^2)")):
    axis.plot(t, y); axis.set_title(name); axis.set_xlabel("s")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_ends():
    "the trajectory starts at 0 and ends at 0.3, at rest at both ends"
    assert abs(x[0]) < 1e-12 and abs(x[-1] - 0.3) < 1e-9 and abs(v[0]) < 1e-3 and abs(v[-1]) < 1e-3

def test_peak():
    "peak velocity occurs at the midpoint and equals 1.875 times the average speed"
    assert abs(t[np.argmax(v)] - 0.3) < 0.005 and abs(v.max() / (0.3 / 0.6) - 1.875) < 0.01

test_ends()
test_peak()
print("All checks passed")

### Exercise: Testing the minimum

Write jerk_cost(x, t): the integral of the squared third derivative of x, computed with np.gradient three times and np.trapezoid (or np.trapz). Then add eps times the bump tau³(1 - tau)³ sin(2π tau) to the minimum-jerk path for eps of 0.01 and 0.02 and compare costs.

In [ ]:
def jerk_cost(x, t):
    j = ...
    return ...

tau = t / T
bump = tau ** 3 * (1 - tau) ** 3 * np.sin(2 * np.pi * tau) * 64        # zero, with two derivatives, at both ends
base_cost = jerk_cost(x, t)
costs = {eps: jerk_cost(x + eps * bump, t) for eps in (0.01, 0.02)}
print(f"minimum-jerk cost {base_cost:.1f}; perturbed: {', '.join(f'{eps}: {c:.1f}' for eps, c in costs.items())}")

In [ ]:
import numpy as np
def test_minimum():
    "every perturbed trajectory has a higher jerk cost"
    assert all(c > base_cost for c in costs.values())

def test_quadratic():
    "doubling the perturbation multiplies the extra cost by about four"
    assert 3.5 < (costs[0.02] - base_cost) / (costs[0.01] - base_cost) < 4.5

test_minimum()
test_quadratic()
print("All checks passed")

### Exercise: Straight paths and a via-point

Compute a direct 2D reach from (0, 0) to (0.3, 0.1) with min_jerk applied to each coordinate, and its largest perpendicular distance from the straight line. Then build a via-point reach by joining two minimum-jerk segments, (0, 0) to the via-point (0.15, 0.12) and on to (0.3, 0), each of duration T/2, and count the peaks of its speed.

In [ ]:
xs, ys = min_jerk(0, 0.3, T, t), min_jerk(0, 0.1, T, t)
d = np.array([0.3, 0.1]) / np.hypot(0.3, 0.1)
straight_dev = ...

# via-point: two joined segments (a simple approximation to the true via-point solution)
th = np.linspace(0, T / 2, 301)
vx = np.r_[min_jerk(0, 0.15, T / 2, th), min_jerk(0.15, 0.3, T / 2, th)[1:]]
vy = np.r_[min_jerk(0, 0.12, T / 2, th), min_jerk(0.12, 0.0, T / 2, th)[1:]]
tt = np.linspace(0, T, len(vx))
speed = np.hypot(np.gradient(vx, tt), np.gradient(vy, tt))
via_miss = np.min(np.hypot(vx - 0.15, vy - 0.12))
n_peaks = ...
print(f"direct reach: largest deviation {straight_dev:.2e} m; via-point reach: miss {via_miss * 1000:.2f} mm, {n_peaks} speed peaks")
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].plot(xs, ys, label="direct"); ax[0].plot(vx, vy, label="via-point"); ax[0].plot(0.15, 0.12, "kx"); ax[0].axis("equal"); ax[0].legend()
ax[1].plot(tt, speed); ax[1].set_xlabel("s"); ax[1].set_ylabel("speed (m/s)"); plt.show()

In [ ]:
import numpy as np
def test_straight():
    "the direct reach deviates from the straight line by less than 1 micrometre"
    assert straight_dev < 1e-6

def test_via():
    "the via-point reach passes within 1 mm of the via-point and has two speed peaks"
    assert via_miss < 1e-3 and n_peaks == 2

test_straight()
test_via()
print("All checks passed")